# Human-in-the-Loop: प्री-एक्शन गेट्स, जोखिम वर्गीकरण, और ऑडिट लॉगिंग

इस पाठ के README में Human-in-the-Loop का परिचय एक छोटे से स्निपेट के साथ दिया गया है जो एजेंट के उत्तर देने के बाद उपयोगकर्ता से `APPROVE` या `REJECT` पूछता है। यह पैटर्न एक अच्छा प्रारंभिक बिंदु है, लेकिन उत्पादन में HITL कार्यान्वयन आम तौर पर तीन अतिरिक्त तत्वों की आवश्यकता होती है:

1. एक **प्री-एक्शन गेट** जो एजेंट के जोखिम भरे कदम से पहले **चलता है**, ताकि लागत, अपरिवर्तनीयता, और विलंबता नियंत्रण में रह सके।
2. **जोखिम वर्गीकरण**, ताकि कम जोखिम वाले क्रियाएं स्वचालित रूप से चलें, मध्यम जोखिम वाले क्रियाएं बैच-अनुमोदित हों, और केवल उच्च जोखिम वाली क्रियाएं मानव के लिए अवरुद्ध करें।
3. एक **ऑडिट लॉग प्लस संशोधन लूप**, ताकि हर गेट निर्णय JSONL के रूप में रिकॉर्ड हो, और अस्वीकृति एजेंट को एक संरचित कारण के साथ पुनः प्रेरित करे बजाय इसके कि सिर्फ `Revising...` प्रिंट हो।

यह नोटबुक `06-system-message-framework.ipynb` की समान प्रिमिटिव्स पर इन सभी को बनाता है। यह `DEMO_MODE = True` में एंड-टू-एंड चलता है (कोई इंटरैक्टिव इनपुट आवश्यक नहीं) या जब `DEMO_MODE = False` हो तो वास्तविक `input()` प्रॉम्प्ट्स के साथ चलता है। नोट: DEMO_MODE में तीसरे लक्ष्य का पुनः प्रयास स्क्रिप्टेड है जिससे लूप मैकेनिक्स एंड-टू-एंड दिखाई देता है। वास्तविक संशोधन-चालित पुनर्वर्गीकरण के लिए `DEMO_MODE = False` और एक ऑपरेटर आवश्यक है।

**क्षेत्र से बाहर (अन्य पाठों में संभाला गया):** प्रामाणिकता और अभिगम नियंत्रण (पाठ 06 README खतरा 2), टूल-कॉल मिडलवेयर (पाठ 14 MAF गहरा विश्लेषण), बहु-एजेंट बहस पैटर्न।

In [ ]:
import json
import os
from datetime import datetime, timezone
from pathlib import Path

from dotenv import load_dotenv
from azure.ai.inference import ChatCompletionsClient
from azure.ai.inference.models import SystemMessage, UserMessage
from azure.core.credentials import AzureKeyCredential

load_dotenv()

DEMO_MODE = True  # set False to use real input() prompts

# Per-run unique log filename so demo runs don't overwrite each other and
# the notebook doesn't touch any pre-existing gate_log.jsonl in the working
# directory.
GATE_LOG_PATH = Path(
    f"gate_log_{datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')}.jsonl"
)

token = os.environ.get("GITHUB_TOKEN", "")
if not token:
    raise RuntimeError(
        "GITHUB_TOKEN environment variable is not set. This notebook needs "
        "a GitHub PAT with 'Models: Read-only' permission. Set GITHUB_TOKEN "
        "in your environment or a local .env file before running."
    )

endpoint = "https://models.github.ai/inference"
model_name = "gpt-4o"

client = ChatCompletionsClient(
    endpoint=endpoint,
    credential=AzureKeyCredential(token),
)


## पैटर्न 1: पूर्व-किया गया गेट

README का HITL स्निपेट पहले एजेंट को कॉल करता है, फिर उपयोगकर्ता से आउटपुट को स्वीकृत करने के लिए कहता है। यह एक **पोस्ट-एक्शन** फ्लो है। एजेंट पहले ही निष्पादित हो चुका होता है, इसलिए LLM कॉल लागत पहले ही चुकायी जा चुकी होती है, और कोई भी साइड इफेक्ट (भेजा गया ईमेल, लिखा गया डेटाबेस पंक्ति, पोस्ट किया गया कमेंट) पहले ही हो चुका होता है।

एक **पूर्व-एक्शन** फ्लो उस गेट को एजेंट के जोखिमपूर्ण चरण को चलाने से पहले डालता है। एजेंट क्रिया का प्रस्ताव करता है, गेट यह निर्णय लेता है कि क्रिया निष्पादित हो या नहीं, और केवल स्वीकृति पर साइड इफेक्ट होता है।

| पहलू | पोस्ट-एक्शन स्वीकृति (README स्निपेट) | पूर्व-एक्शन गेट (यह नोटबुक) |
|---|---|---|
| स्वीकृति कब चलती है? | एजेंट के आउटपुट देने के बाद | किसी भी साइड-इफेक्ट के निष्पादन से पहले |
| अस्वीकृति पर LLM लागत | पहले ही चुकाई गई | केवल प्रस्ताव के लिए चुकाई गई, क्रिया के लिए नहीं |
| अपरिवर्तनीय साइड इफेक्ट | संभावित (क्रिया पहले ही हो चुकी है) | रोका गया |
| ऑडिट स्पष्टता | स्वीकृति एक प्रिंट स्टेटमेंट है | स्वीकृति एक JSON रिकॉर्ड है जिसमें टाइमस्टैम्प, क्रिया, कारण होता है |


In [ ]:
def gate_action(action_description: str, risk_tier: str, attempt: int = 0) -> dict:
    """Run a single pre-action gate.

    Returns a decision dict with keys: decision, reason, ts.
    Decision is one of: approve, deny, escalate.
    Safe default on EOF or unexpected input is deny.

    DEMO_MODE behavior: high-risk actions are denied on attempt 0 and
    auto-approved on attempt >= 1. This is scripted approval to show the
    loop mechanics (deny -> retry -> approve). It is NOT revision-driven
    re-classification. Real revision-driven re-classification requires
    DEMO_MODE=False and a human operator who evaluates the revised
    proposal on its own merits.
    """
    print(f"[gate] proposed action ({risk_tier}, attempt={attempt}): {action_description}")

    if DEMO_MODE:
        if risk_tier == "high":
            decision = "approve" if attempt >= 1 else "deny"
            reason = (
                "DEMO_MODE: scripted approval on retry to show loop mechanics"
                if attempt >= 1
                else "DEMO_MODE: high risk denied on first attempt"
            )
        else:
            decision = "approve"
            reason = f"DEMO_MODE canned response for tier={risk_tier}"
    else:
        try:
            raw = input("[gate] approve / deny / escalate? ").strip().lower()
        except EOFError:
            raw = ""
        if raw in {"approve", "deny", "escalate"}:
            decision, reason = raw, "operator input"
        elif raw == "":
            decision, reason = "deny", "no input received, defaulted to deny"
        else:
            decision, reason = "deny", f"invalid input {raw!r}, defaulted to deny"

    return {
        "decision": decision,
        "reason": reason,
        "action": action_description,
        "risk_tier": risk_tier,
        "ts": datetime.now(timezone.utc).isoformat(),
    }


## पैटर्न 2: जोखिम वर्गीकरण

हर क्रिया को मानव अनुमोदन की आवश्यकता नहीं होती है। एक सार्वजनिक API के खिलाफ केवल पढ़ने वाला लुकअप ग्राहक को ईमेल भेजने की तुलना में अलग स्तर पर होता है। दोनों को समान मानना ऑपरेटर का ध्यान बर्बाद करता है और एजेंट को धीमा कर देता है।

एक सरल 3-स्तरीय मॉडल:

| स्तर | उदाहरण | अनुमोदन प्रवाह |
|---|---|---|
| `low` (केवल-पढ़ने वाला) | एक ज्ञान आधार खोजें, उड़ान विकल्प देखें, एक सार्वजनिक वेब पृष्ठ प्राप्त करें | स्वचालित निष्पादन, लेखा परीक्षा के लिए लॉग किया गया |
| `medium` (सस्ती परिवर्तन) | परिणाम कैश करें, काउंटर बढ़ाएं, एक अनुस्मारक निर्धारित करें | स्वचालित निष्पादन, लेकिन दैनिक समीक्षा के लिए बैच में |
| `high` (बाहरी-सामना वाला या अपरिवर्तनीय) | ईमेल भेजें, कार्ड चार्ज करें, सार्वजनिक चैनल पर पोस्ट करें | मानव अनुमोदन पर ब्लॉक करें |

यह एक वर्गीकरण है। उत्पादन प्रणालियाँ अक्सर अधिक सूक्ष्म स्तरों का उपयोग करती हैं (जैसे, AWS IAM अनुमति स्तर, भूमिका-आधारित पहुँच स्तर)। नीचे दिया गया 3-स्तरीय संस्करण एक एजेंट के लिए सबसे छोटा उपयोगी संस्करण है जो केवल-पढ़ने वाले और साइड-इफेक्टिंग कार्यों को मिलाता है।

नीचे क्लासिफायर कीवर्ड हीयूरिस्टिक्स का उपयोग करता है ताकि डेमो निश्चित और सस्ता बना रहे। एक उत्पादन प्रणाली में आप इसे एक सीखे हुए क्लासिफायर या नीति इंजन से बदलेंगे।

In [ ]:
LOW_RISK_KEYWORDS = {
    "look", "lookup", "search", "fetch", "read", "query", "view",
    "get", "list", "weather", "summarize",
}
HIGH_RISK_KEYWORDS = {
    "send", "email", "post", "publish", "charge", "pay", "transfer",
    "delete", "drop", "cancel", "refund",
}
MEDIUM_RISK_KEYWORDS = {
    "cache", "schedule", "reminder", "book", "reserve", "update",
    "increment", "log",
}

AUTO_APPROVE_REASONS = {
    "low": "auto-approved (low risk)",
    "medium": "auto-approved (medium risk, queued for batched review)",
}


def classify_risk(action: str) -> str:
    """Classify an action string into one of: low, medium, high.

    Keyword-based heuristic. Checks high-risk first (most severe), then
    low-risk explicit reads, then medium-risk mutations. Unrecognized
    actions default to medium, not low.

    Default for unrecognized actions is 'medium', not 'low'. A read-only
    keyword set will always have blind spots, and the parent README's
    threat list (critical-system access, knowledge-base poisoning,
    cascading errors) all involve cases an action-name alone cannot rule
    out. Routing unknown actions through batched review is the safer
    default than auto-executing them.
    """
    text = action.lower()
    if any(kw in text for kw in HIGH_RISK_KEYWORDS):
        return "high"
    if any(kw in text for kw in LOW_RISK_KEYWORDS):
        return "low"
    if any(kw in text for kw in MEDIUM_RISK_KEYWORDS):
        return "medium"
    # Explicit fail-safe default: unrecognized actions route to batched review.
    return "medium"


def tiered_gate(action: str, attempt: int = 0) -> dict:
    """Classify then gate. Low and medium tiers auto-approve; high blocks."""
    tier = classify_risk(action)
    if tier in AUTO_APPROVE_REASONS:
        return {
            "decision": "approve",
            "reason": AUTO_APPROVE_REASONS[tier],
            "action": action,
            "risk_tier": tier,
            "ts": datetime.now(timezone.utc).isoformat(),
        }
    return gate_action(action, tier, attempt=attempt)


## पैटर्न 3: ऑडिट लॉग और संशोधन लूप

एक `print("Response approved.")` एक ऑडिट लॉग नहीं है। विश्वास के लिए, हर गेट निर्णय को एक संरचित घटना के रूप में रिकॉर्ड किया जाना चाहिए जिसे आप बाद में क्वेरी कर सकते हैं, दोहरा सकते हैं, या किसी घटना समीक्षा से जोड़ सकते हैं।

दो हिस्से:

1. **सिर्फ़ जोड़ने योग्य JSONL।** प्रत्येक निर्णय के लिए एक पंक्ति, जिसमें टाइमस्टैम्प, क्रिया, स्तर, निर्णय, कारण होता है। grep करना आसान, बाद में वास्तविक लॉग स्टोर को भेजना आसान।
2. **अस्वीकृति पर संशोधन लूप।** जब गेट `deny` लौटाता है, तो एजेंट अपने आप को अस्वीकृति कारण के साथ संदर्भ में फिर से प्रॉम्प्ट करता है, ताकि अगली प्रस्तुति समस्या से बच सके।

In [ ]:
def log_decision(decision: dict) -> None:
    """Append a gate decision to the JSONL audit log."""
    with GATE_LOG_PATH.open("a", encoding="utf-8") as f:
        f.write(json.dumps(decision) + "\n")


def propose_action(goal: str, prior_rejection: str | None = None) -> str:
    """Ask the LLM to propose a concrete next action for a goal.

    If prior_rejection is provided, it is fed back so the LLM can avoid
    the same failure mode in the next proposal.
    """
    system = (
        "You are an action planner for an agent. Propose ONE concrete next\n"
        "action (a single sentence) toward the user's goal. If a prior\n"
        "rejection reason is given, propose a different action that addresses\n"
        "the rejection."
    )
    user_text = f"Goal: {goal}"
    if prior_rejection:
        user_text += f"\n\nPrior proposal was denied. Reason: {prior_rejection}"

    response = client.complete(
        model=model_name,
        messages=[SystemMessage(content=system), UserMessage(content=user_text)],
    )
    return response.choices[0].message.content.strip()


def run_with_revision(goal: str, max_revisions: int = 2) -> dict:
    """Propose, gate, and on rejection revise up to max_revisions times."""
    prior_reason: str | None = None
    for attempt in range(max_revisions + 1):
        action = propose_action(goal, prior_rejection=prior_reason)
        decision = tiered_gate(action, attempt=attempt)
        decision["attempt"] = attempt
        log_decision(decision)
        if decision["decision"] == "approve":
            return decision
        prior_reason = decision["reason"]
    return {**decision, "final": "max_revisions_reached"}


In [ ]:
# End-to-end demo: three goals at three different risk profiles.
# GATE_LOG_PATH is per-run (timestamped) so no prior log is touched.

goals = [
    "Look up the weather in Seattle for the customer's trip planning.",
    "Schedule a reminder for the customer to check in 24 hours before their flight.",
    "Send a marketing email to the customer about premium upgrade options.",
]

for goal in goals:
    print(f"\n=== Goal: {goal} ===")
    outcome = run_with_revision(goal, max_revisions=1)
    print(f"[final] {outcome['decision']} ({outcome['reason']})")

print(f"\n=== Audit log ({GATE_LOG_PATH.name}) ===")
for line in GATE_LOG_PATH.read_text(encoding="utf-8").splitlines():
    record = json.loads(line)
    print(f"  [{record['risk_tier']:6s}] {record['decision']:8s} "
          f"attempt={record.get('attempt', '?')} action={record['action'][:140]}")


## अतिरिक्त संसाधन

कई अन्य सार्वजनिक परियोजनाएँ इन HITL पैटर्न के विभिन्न संस्करणों को लागू करती हैं। यह पता लगाने के लिए तरीकों की तुलना करें कि आपकी स्टैक के लिए क्या उपयुक्त है:

- **LangChain** मानव-इन-द-लूप टूल रैपर ([docs](https://python.langchain.com/docs/integrations/tools/human_tools)): ऐसे टूल रैपर जो मानव इनपुट के लिए निष्पादन को रोकते हैं।
- **AutoGen** `UserProxyAgent` ([v0.2 docs](https://microsoft.github.io/autogen/0.2/docs/topics/human-in-the-loop); AutoGen v0.4+ ने इसे पुनर्गठित किया): विशेष रूप से मानव प्रतिनिधित्व के लिए एक एजेंट भूमिका का उपयोग करता है जो मल्टी-एजेंट संवादों में होता है।
- **Semantic Kernel** फ़ंक्शन फ़िल्टर ([docs](https://learn.microsoft.com/en-us/semantic-kernel/concepts/enterprise-readiness/filters)): प्रत्येक फ़ंक्शन कॉल के चारों ओर चलने वाला मिडलवेयर, जो गेटिंग लॉजिक के लिए उपयुक्त है।

प्रत्येक परियोजना तीन उप-पैटर्न को अलग-अलग तरीके से संभालती है: LangChain उन्हें टूल के रूप में रैप करता है, AutoGen एजेंट भूमिका का उपयोग करता है, Semantic Kernel मिडलवेयर फ़िल्टर का उपयोग करता है। अपनी एजेंट के लिए डिजाइन चुनने से पहले एक या दो कार्यान्वयन को अंत से अंत तक पढ़ें।

---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**अस्वीकरण**:
इस दस्तावेज़ का अनुवाद AI अनुवाद सेवा [Co-op Translator](https://github.com/Azure/co-op-translator) का उपयोग करके किया गया है। जबकि हम सटीकता के लिए प्रयास करते हैं, कृपया ध्यान दें कि स्वचालित अनुवादों में त्रुटियाँ या अशुद्धियाँ हो सकती हैं। मूल दस्तावेज़ अपनी मूल भाषा में ही प्रामाणिक स्रोत माना जाना चाहिए। महत्वपूर्ण जानकारी के लिए, पेशेवर मानव अनुवाद की सिफारिश की जाती है। इस अनुवाद के उपयोग से उत्पन्न किसी भी गलतफहमी या गलत व्याख्या के लिए हम उत्तरदायी नहीं हैं।
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
